# Notebook 02 — AI Reviewer

**Purpose:** Evaluate a student's work submission and return structured feedback with scores.

When a student submits their completed task, this pipeline evaluates:
- Code quality (for coding tasks)
- Communication quality (for emails, reports, meeting notes)
- Problem-solving approach
- Time management (did they meet the deadline?)
- Completeness (did they address all requirements?)

**AI Model:** Groq — `llama-3.3-70b-versatile` via `langchain-groq`

**Key Techniques:**
- `StateGraph` (LangGraph) for the review workflow
- Evaluation rubrics loaded from `data/evaluation_rubrics/`
- Chain-of-thought reasoning before scoring
- `ChatGroq` with JSON mode for structured output
- XP formula: `base_xp * (score/100) * difficulty_multiplier`

**Input:** `{ task, submission }`

**Output:** Structured JSON with scores, feedback, XP earned

---
## 1. Setup & Dependencies

In [ ]:
# %pip install -r requirements.txt

In [ ]:
import os
import json
import time
from datetime import datetime
from typing import TypedDict, Optional
from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, END

load_dotenv()

print("All imports loaded")
print(f"Groq API key present: {bool(os.getenv('GROQ_API_KEY'))}")

In [ ]:
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in .env file")

MODEL_NAME = "llama-3.3-70b-versatile"

model = ChatGroq(
    model=MODEL_NAME,
    temperature=0.3,
    max_tokens=2000,
    api_key=GROQ_API_KEY,
)

print(f"ChatGroq initialized with model: {MODEL_NAME}")

---
## 2. Define Review State (LangGraph)

In [ ]:
class ReviewState(TypedDict):
    task: dict
    submission: dict
    rubric: Optional[dict]
    system_prompt: str
    user_message: str
    result: Optional[dict]
    xp_earned: Optional[int]
    error: Optional[str]
    response_time: Optional[float]
    generated_at: Optional[str]

print("ReviewState defined")

In [ ]:
DIFFICULTY_MULTIPLIER = {
    "easy": 0.5,
    "intermediate": 1.0,
    "medium": 1.0,
    "hard": 2.0,
}

BASE_XP = 200

OUTPUT_SCHEMA = {
    "overall_score": "int 0-100",
    "scores": {
        "code_quality": "int 0-100",
        "communication": "int 0-100",
        "problem_solving": "int 0-100",
        "time_management": "int 0-100",
        "completeness": "int 0-100",
    },
    "feedback": "string - detailed evaluation feedback",
    "strengths": ["string - list of strengths"],
    "improvement_suggestions": ["string - list of improvements"],
    "what_a_senior_would_say": "string - human-feel advice line",
}

print("Output schema defined")

---
## 3. Load Evaluation Rubrics

In [ ]:
DATA_DIR = os.path.join(os.path.dirname(os.path.abspath("")), "data")

def load_json(filepath):
    full_path = os.path.join(DATA_DIR, filepath)
    if not os.path.exists(full_path):
        print(f"Warning: {full_path} not found.")
        return {}
    with open(full_path, "r", encoding="utf-8") as f:
        return json.load(f)

code_rubric = load_json("evaluation_rubrics/code_review_rubric.json")
comm_rubric = load_json("evaluation_rubrics/communication_rubric.json")
design_rubric = load_json("evaluation_rubrics/design_rubric.json")

print(f"Loaded: code_review_rubric ({len(code_rubric.get('categories', []))} categories)")
print(f"Loaded: communication_rubric ({len(comm_rubric.get('categories', []))} categories)")
print(f"Loaded: design_rubric ({len(design_rubric.get('categories', []))} categories)")

---
## 4. Build LangGraph Review Nodes

In [ ]:
def load_rubrics_node(state: ReviewState) -> dict:
    task_type = state["task"].get("task_type", "coding")

    if task_type in ("coding", "bug_fix"):
        rubric = code_rubric
    elif task_type in ("report", "meeting"):
        rubric = comm_rubric
    elif task_type == "design":
        rubric = design_rubric
    else:
        rubric = code_rubric

    print(f"  [load_rubrics] Using rubric: {rubric.get('rubric_name', 'unknown')} for task_type: {task_type}")
    return {"rubric": rubric}

print("load_rubrics_node defined")

In [ ]:
def build_review_prompts_node(state: ReviewState) -> dict:
    task = state["task"]
    submission = state["submission"]
    rubric = state["rubric"]

    rubric_json = json.dumps(rubric, indent=2) if rubric else "{}"
    task_json = json.dumps(task, indent=2)
    submission_content = submission.get("content", "")
    submitted_at = submission.get("submitted_at", "unknown")
    deadline_hours = task.get("deadline_hours", 0)

    submitted_dt = None
    if submitted_at != "unknown":
        try:
            submitted_dt = datetime.fromisoformat(submitted_at)
        except:
            pass

    time_management_note = ""
    if submitted_dt and deadline_hours:
        time_management_note = (
            f"The task had a deadline of {deadline_hours} hours. "
            f"The student submitted at {submitted_at}. "
            "Evaluate if they met the deadline and managed their time well."
        )

    system_lines = [
        "You are a senior technical reviewer at a mid-size tech company.",
        "You have 8+ years of experience reviewing code, designs, and communications.",
        "",
        "Your job is to evaluate student work submissions fairly and provide constructive feedback.",
        "",
        "## Evaluation Rubric",
        rubric_json,
        "",
        "## Task Details",
        task_json,
        "",
    ]
    if time_management_note:
        system_lines.append("## Time Context")
        system_lines.append(time_management_note)

    system_lines.append("")
    system_lines.append("## Chain of Thought Instructions")
    system_lines.append("Before scoring, reason step by step:")
    system_lines.append("1. First, analyze what the task required")
    system_lines.append("2. Then, examine what the student submitted")
    system_lines.append("3. Compare against each rubric category")
    system_lines.append("4. Score each category 0-100 based on the rubric levels")
    system_lines.append("5. Calculate the overall score as a weighted average of all categories")
    system_lines.append("6. Provide specific, actionable feedback for each area")
    system_lines.append("")
    system_lines.append("Return your evaluation as a valid JSON object matching the output schema.")

    user_lines = [
        "## Student Submission",
        "The student submitted the following work:",
        "```",
        submission_content,
        "```",
    ]

    return {
        "system_prompt": "\n".join(system_lines),
        "user_message": "\n".join(user_lines),
    }

print("build_review_prompts_node defined")

In [ ]:
def evaluate_node(state: ReviewState) -> dict:
    messages = [
        SystemMessage(content=state["system_prompt"]),
        HumanMessage(content=state["user_message"]),
    ]

    start = time.time()
    try:
        response = model.bind(response_format={"type": "json_object"}).invoke(messages)
        elapsed = time.time() - start
        content = response.content

        result = json.loads(content)

        required_keys = ["overall_score", "scores", "feedback", "strengths", "improvement_suggestions", "what_a_senior_would_say"]
        for key in required_keys:
            if key not in result:
                raise ValueError(f"Response missing required field: {key}")

        score_keys = ["code_quality", "communication", "problem_solving", "time_management", "completeness"]
        for key in score_keys:
            if key not in result.get("scores", {}):
                raise ValueError(f"Response missing score field: {key}")

        print(f"  [evaluate] Overall score: {result['overall_score']}, Time: {elapsed:.2f}s")
        return {
            "result": result,
            "response_time": round(elapsed, 2),
            "generated_at": datetime.now().isoformat(),
            "error": None,
        }
    except Exception as e:
        print(f"  [evaluate] Failed: {e}")
        return {"error": str(e)}

print("evaluate_node defined")

In [ ]:
def calculate_xp_node(state: ReviewState) -> dict:
    result = state.get("result")
    task = state["task"]
    if not result:
        return {"xp_earned": 0}

    overall = result.get("overall_score", 0)
    difficulty = task.get("difficulty", "medium")
    multiplier = DIFFICULTY_MULTIPLIER.get(difficulty, 1.0)

    xp = int(BASE_XP * (overall / 100) * multiplier)

    result["xp_earned"] = xp

    print(f"  [calculate_xp] Score: {overall}, Difficulty: {difficulty} (x{multiplier}), XP: {xp}")
    return {"xp_earned": xp}

print("calculate_xp_node defined")

---
## 5. Compile the LangGraph

In [ ]:
def build_review_graph():
    workflow = StateGraph(ReviewState)

    workflow.add_node("load_rubrics", load_rubrics_node)
    workflow.add_node("build_prompts", build_review_prompts_node)
    workflow.add_node("evaluate", evaluate_node)
    workflow.add_node("calculate_xp", calculate_xp_node)

    workflow.set_entry_point("load_rubrics")
    workflow.add_edge("load_rubrics", "build_prompts")
    workflow.add_edge("build_prompts", "evaluate")
    workflow.add_edge("evaluate", "calculate_xp")
    workflow.add_edge("calculate_xp", END)

    return workflow.compile()

app = build_review_graph()
print("LangGraph AI Reviewer compiled successfully")

---
## 6. Helper Functions

In [ ]:
def review_submission(task: dict, submission: dict, max_retries=2):
    last_error = None
    for attempt in range(max_retries):
        state = ReviewState(
            task=task,
            submission=submission,
            rubric=None,
            system_prompt="",
            user_message="",
            result=None,
            xp_earned=None,
            error=None,
            response_time=None,
            generated_at=None,
        )
        try:
            final_state = app.invoke(state)
            if final_state.get("error"):
                raise RuntimeError(final_state["error"])

            result = final_state["result"]
            result["_metadata"] = {
                "model": MODEL_NAME,
                "framework": "langchain + langgraph",
                "response_time_seconds": final_state.get("response_time", 0),
                "generated_at": final_state.get("generated_at", ""),
            }
            return result

        except Exception as e:
            last_error = str(e)
            print(f"Attempt {attempt + 1}/{max_retries} failed: {last_error}")

    raise RuntimeError(f"All attempts failed. Last error: {last_error}")

print("review_submission function defined")

In [ ]:
def display_review(result):
    print("=" * 70)
    print("AI REVIEW RESULTS")
    print("=" * 70)

    print(f"\nOVERALL SCORE: {result.get('overall_score', 'N/A')}/100")
    print(f"XP EARNED: {result.get('xp_earned', 'N/A')}")
    print()

    scores = result.get("scores", {})
    categories = [
        ("Code Quality", "code_quality"),
        ("Communication", "communication"),
        ("Problem Solving", "problem_solving"),
        ("Time Management", "time_management"),
        ("Completeness", "completeness"),
    ]
    for label, key in categories:
        score = scores.get(key, "N/A")
        bar = "#" * (score // 5) if isinstance(score, int) else ""
        print(f"  {label:20s}: {score:>3d} {bar}")

    print(f"\n{'=' * 70}")
    print("FEEDBACK")
    print(f"{'=' * 70}")
    print(result.get("feedback", "N/A"))

    print(f"\nSTRENGTHS")
    print(f"{'=' * 70}")
    for s in result.get("strengths", []):
        print(f"  + {s}")

    print(f"\nIMPROVEMENT SUGGESTIONS")
    print(f"{'=' * 70}")
    for s in result.get("improvement_suggestions", []):
        print(f"  - {s}")

    print(f"\n{'=' * 70}")
    print(f"WHAT A SENIOR WOULD SAY:")
    print(result.get("what_a_senior_would_say", "N/A"))
    print(f"{'=' * 70}")

    meta = result.get("_metadata", {})
    print(f"\nModel: {meta.get('model', 'N/A')}")
    print(f"Response time: {meta.get('response_time_seconds', 'N/A')}s")

print("display_review function defined")

---
## 7. Tests

### Test 1: Good solution vs bad solution — verify scores differ

In [ ]:
print("=" * 70)
print("TEST 1: Good solution vs Bad solution")
print("=" * 70)

task = {
    "task_id": "task_001",
    "title": "Fix Authentication Token Expiry Bug",
    "description": "Users are being logged out prematurely. Fix the JWT token expiry handling.",
    "task_type": "bug_fix",
    "difficulty": "medium",
    "deadline_hours": 4,
    "expected_deliverable": "Code fix with token refresh logic",
}

good_submission = {
    "content": '''
I implemented a token refresh mechanism that solves the premature logout issue:

1. Added an axios interceptor that catches 401 responses
2. When a 401 is received, the interceptor calls /api/auth/refresh with the refresh token
3. If refresh succeeds, the original request is retried with the new token
4. If refresh fails, the user is redirected to login

Key code:
```javascript
axios.interceptors.response.use(
  response => response,
  async error => {
    const originalRequest = error.config;
    if (error.response.status === 401 && !originalRequest._retry) {
      originalRequest._retry = true;
      try {
        const { data } = await axios.post('/api/auth/refresh', {
          refreshToken: localStorage.getItem('refreshToken')
        });
        localStorage.setItem('accessToken', data.accessToken);
        originalRequest.headers.Authorization = `Bearer ${data.accessToken}`;
        return axios(originalRequest);
      } catch (refreshError) {
        localStorage.clear();
        window.location.href = '/login';
        return Promise.reject(refreshError);
      }
    }
    return Promise.reject(error);
  }
);
```

I also added proper error handling for network failures and token expiry edge cases.
The solution handles the refresh token rotation pattern recommended by security best practices.
''',
    "submitted_at": datetime.now().isoformat(),
    "deadline": (datetime.now().isoformat()),
}

bad_submission = {
    "content": '''
I tried to fix the login issue. I think the problem is with the token. I just increased the token expiry time to 24 hours in the backend config. That should fix it.
''',
    "submitted_at": datetime.now().isoformat(),
    "deadline": (datetime.now().isoformat()),
}

good_result = None
bad_result = None

print("\n--- Evaluating GOOD submission ---")
try:
    good_result = review_submission(task, good_submission)
    display_review(good_result)
except Exception as e:
    print(f"Failed: {e}")

print("\n\n--- Evaluating BAD submission ---")
try:
    bad_result = review_submission(task, bad_submission)
    display_review(bad_result)
except Exception as e:
    print(f"Failed: {e}")

if good_result and bad_result:
    print("\n" + "=" * 70)
    print("COMPARISON")
    print(f"Good solution score: {good_result.get('overall_score', 0)}")
    print(f"Bad solution score: {bad_result.get('overall_score', 0)}")
    diff = good_result.get('overall_score', 0) - bad_result.get('overall_score', 0)
    if diff > 30:
        print(f"PASS: Scores differ meaningfully ({diff} point gap)")
    else:
        print(f"WARNING: Score gap is small ({diff} points)")

### Test 2: Empty submission — check graceful handling

In [ ]:
print("=" * 70)
print("TEST 2: Empty submission")
print("=" * 70)

empty_submission = {
    "content": "",
    "submitted_at": datetime.now().isoformat(),
}

try:
    result = review_submission(task, empty_submission)
    display_review(result)

    if result.get("overall_score", 100) < 30:
        print("\nPASS: Empty submission correctly scored low")
    else:
        print(f"\nWARNING: Empty submission scored {result.get('overall_score')}")
except Exception as e:
    print(f"Failed: {e}")

### Test 3: Code with subtle bugs — check if AI catches them

In [ ]:
print("=" * 70)
print("TEST 3: Code with subtle bugs")
print("=" * 70)

buggy_submission = {
    "content": '''function getUserData(userId) {
  const response = fetch(`/api/users/${userId}`);
  const data = response.json();
  return data;
}

function saveUser(user) {
  const res = fetch('/api/users', {
    method: 'post',
    body: JSON.stringify(user),
  });
  return res;
}
''',
    "submitted_at": datetime.now().isoformat(),
}

buggy_task = {
    "task_id": "task_api_001",
    "title": "Implement User API Functions",
    "description": "Write functions to fetch and save user data from the API.",
    "task_type": "coding",
    "difficulty": "easy",
    "deadline_hours": 2,
}

try:
    result = review_submission(buggy_task, buggy_submission)
    display_review(result)

    feedback_lower = result.get("feedback", "").lower()
    bugs_mentioned = any(word in feedback_lower for word in ["await", "async", "promise", "missing", "error"])
    print(f"\nBugs detected in feedback: {bugs_mentioned}")
except Exception as e:
    print(f"Failed: {e}")

### Test 4: Well-written vs poorly-written email

In [ ]:
print("=" * 70)
print("TEST 4: Communication scoring - Email quality")
print("=" * 70)

email_task = {
    "task_id": "task_report_001",
    "title": "Weekly Progress Report",
    "description": "Write a weekly progress report email to the team lead.",
    "task_type": "report",
    "difficulty": "easy",
    "deadline_hours": 24,
}

good_email = {
    "content": '''Subject: Weekly Progress Report - Week 3

Hi Priya,

Here's my progress report for this week:

Completed Tasks:
1. Fixed the authentication token expiry bug (completed with token refresh mechanism)
2. Built the data table component with sorting and pagination
3. Wrote integration tests for the auth endpoints

Blockers:
- Waiting for design approval on the file upload component
- Need access to the staging database for performance testing

Next Week's Plan:
1. Implement the file upload component
2. Start working on the email notification service
3. Code review for team members' PRs

Please let me know if you need any additional details.

Best regards,
[Student Name]
''',
    "submitted_at": datetime.now().isoformat(),
}

bad_email = {
    "content": "hey i did some work this week fixed some bugs and worked on stuff. will do more next week. thanks",
    "submitted_at": datetime.now().isoformat(),
}

good_result = None
bad_result = None

print("\n--- Evaluating GOOD email ---")
try:
    good_result = review_submission(email_task, good_email)
    display_review(good_result)
except Exception as e:
    print(f"Failed: {e}")

print("\n--- Evaluating POOR email ---")
try:
    bad_result = review_submission(email_task, bad_email)
    display_review(bad_result)
except Exception as e:
    print(f"Failed: {e}")

if good_result and bad_result:
    print("\n" + "=" * 70)
    print("COMMUNICATION SCORE COMPARISON")
    gs = good_result.get("scores", {}).get("communication", 0)
    bs = bad_result.get("scores", {}).get("communication", 0)
    print(f"Good email communication score: {gs}")
    print(f"Poor email communication score: {bs}")
    if gs > bs + 20:
        print("PASS: Communication scores differ meaningfully")
    else:
        print(f"WARNING: Small gap ({gs - bs} points)")

### Test 5: Verify XP formula

In [ ]:
print("=" * 70)
print("TEST 5: XP Formula Verification")
print("=" * 70)

print("Formula: XP = BASE_XP * (overall_score / 100) * difficulty_multiplier")
print(f"BASE_XP = {BASE_XP}")
print(f"Difficulty multipliers: {DIFFICULTY_MULTIPLIER}")
print()

test_cases = [
    {"task": {"difficulty": "easy"}, "score": 80, "expected_xp": int(BASE_XP * 0.80 * 0.5)},
    {"task": {"difficulty": "medium"}, "score": 75, "expected_xp": int(BASE_XP * 0.75 * 1.0)},
    {"task": {"difficulty": "hard"}, "score": 90, "expected_xp": int(BASE_XP * 0.90 * 2.0)},
]

for tc in test_cases:
    diff = tc["task"]["difficulty"]
    score = tc["score"]
    multi = DIFFICULTY_MULTIPLIER.get(diff, 1.0)
    xp = int(BASE_XP * (score / 100) * multi)
    print(f"  {diff:10s} | score: {score:2d} | multiplier: {multi:.1f} | XP: {xp} | expected: {tc['expected_xp']}")
    assert xp == tc["expected_xp"], f"XP mismatch!"

print("\nPASS: All XP calculations match expected values")

---
## 8. Summary

### Test Results Checklist
- [ ] **Test 1:** Good solution vs bad solution — scores differ meaningfully
- [ ] **Test 2:** Empty submission — graceful handling
- [ ] **Test 3:** Code with subtle bugs — AI catches them
- [ ] **Test 4:** Well-written vs poorly-written email — communication scoring works
- [ ] **Test 5:** XP formula verification

### Architecture
```
StateGraph(ReviewState)
  [load_rubrics] -> [build_prompts] -> [evaluate] -> [calculate_xp] -> END
       |                 |                  |               |
  Selects rubric     Builds prompt      Calls ChatGroq   Computes XP
  by task_type       with rubrics +      with CoT +      = BASE * score
                     task context        JSON mode        * difficulty
```

### Scoring Categories
| Category | Weight | Evaluated For |
|----------|--------|--------------|
| code_quality | 25% | Coding, bug_fix tasks |
| communication | 25% | All task types |
| problem_solving | 25% | All task types |
| time_management | 0% | Based on deadline compliance |
| completeness | 25% | All task types |